# STEP9. 산업 재무특성 데이터 수집 (부채비율·배당수익률·의약품 R&D)

## 분석 질문

- H3 탐색에 필요한 산업별 부채비율·배당수익률과 의약품 R&D를 재현 가능한 절차로 수집할 수 있는가?
- 관련 가설: H3 — Baseline 공동·쌍별검정에서 확인된 산업별 금리베타 차이(의약품 대 나머지 6개 산업의 대비에 집중)가 산업 재무특성 순위와 정렬되는가? (검정은 STEP10)

## 입력

| 구분 | 파일 / API | 설명 |
|---|---|---|
| 파일 | `data/raw/firm/krx_manual/industry_constituents.csv` | KRX 지수 구성종목 스냅샷(2025) |
| 파일 | `data/raw/firm/krx_manual/krx_top10_mktcap_2024.csv` | 시총 가중치(2024년말) |
| 파일 | `data/raw/firm/krx_manual/rnd_pharma_capitalized.csv` | 의약품 자본화 기업 R&D/매출(사업보고서) |
| API | DART OpenAPI 재무제표·배당 | 부채비율·매출·경상연구개발비·배당수익률 |
| 이전 STEP | 없음 | — |

## 출력

| 구분 | 파일 | 설명 |
|---|---|---|
| Table | `data/raw/firm/firm_financials.csv` | 기업×연도 부채비율·배당수익률·매출·R&D |
| Table | `data/raw/firm/selected_top10.csv` | 산업별 선정 10종목 + 시총가중치 |
| Table | `data/raw/firm/collection_errors.csv` | 수집 오류 원장 |
| Table | `data/processed/pharma_rnd_support.csv` | 의약품 R&D/매출 보조근거 |
| Summary | `results/H3_firm_characteristics/summary.md` | (작성 예정) |


In [1]:
# [셀 1] 설정 · 경로 · API 키 (import·경로·상수는 여기서 한 번만)

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
import sys
from pathlib import Path
_anchors = []
if "__vsc_ipynb_file__" in globals():
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))
import config
if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(f"의도하지 않은 config.py: {config.__file__}")
# --- 부트스트랩 끝 ---

import os, re, time, json, zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import numpy as np, pandas as pd, requests
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
config.setup_korean_font()
try:
    from dotenv import load_dotenv; load_dotenv(config.PROJECT_ROOT / ".env")
except ImportError:
    pass

API_KEY = os.getenv("OPENDART_API_KEY") or os.getenv("DART_API_KEY")
assert API_KEY, ("부채비율·배당수익률·R&D 수집에는 OPENDART_API_KEY가 필요합니다. "
                 "opendart.fss.or.kr에서 무료 발급 후 .env에 OPENDART_API_KEY=... 추가.")

RAW_FIRM_DIR = config.RAW_DIR / "firm"
KRX_MANUAL_DIR = config.KRX_MANUAL_DIR
DART_RAW_DIR = RAW_FIRM_DIR / "dart_responses"
PROCESSED_DIR = config.PROCESSED_DIR
for d in (RAW_FIRM_DIR, DART_RAW_DIR, PROCESSED_DIR, KRX_MANUAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

INDUSTRIES = ["건설업", "금융업", "운수장비", "의약품", "전기전자", "철강금속", "화학"]
YEARS = list(range(2016, 2026))
DIV_MISSING_AS_ZERO = True   # 사업보고서 있으나 현금배당 공시 없으면 배당수익률=0(무배당)
print(f"[환경] DART 키={'설정' if API_KEY else '미설정'} · 산업 {len(INDUSTRIES)}개 · {YEARS[0]}~{YEARS[-1]}")


그래프 폰트: Malgun Gothic
[환경] DART 키=설정 · 산업 7개 · 2016~2025


## 구성종목·가중치 입력

### 목적

- KRX 지수 구성종목 스냅샷과 시총 가중치를 로드하고 식별자·산업·중복을 검증한다.
- 구성종목은 KOSPI 산업별 지수(2025 스냅샷) 시총 상위 10종목이며, KRX 정보데이터시스템에서 확보한 버전관리 입력이다.


In [2]:
# [셀 2] 구성종목·시총 로드 + 검증
CONST_CSV = KRX_MANUAL_DIR / "industry_constituents.csv"
MCAP_CSV = KRX_MANUAL_DIR / "krx_top10_mktcap_2024.csv"
assert CONST_CSV.exists(), f"[필수입력 없음] {CONST_CSV} — KRX 지수 구성종목 스냅샷 필요"
con = pd.read_csv(CONST_CSV, dtype={"stock_code": str})
con["stock_code"] = con["stock_code"].str.zfill(6)
assert set(con["industry"].unique()) <= set(INDUSTRIES), "허용되지 않은 산업명"
assert (con["stock_code"].str.fullmatch(r"\d{6}")).all(), "종목코드 6자리 아님"
assert not con.duplicated("stock_code").any(), "종목코드 중복"
multi = con.groupby("stock_code")["industry"].nunique()
assert (multi <= 1).all(), "한 종목이 복수 산업에 배정됨"

if MCAP_CSV.exists():
    mcap = pd.read_csv(MCAP_CSV, dtype={"stock_code": str})[["stock_code", "market_cap_2024"]]
    mcap["stock_code"] = mcap["stock_code"].str.zfill(6)
    con = con.merge(mcap, on="stock_code", how="left")
else:
    con["market_cap_2024"] = np.nan
    print(f"[경고] {MCAP_CSV} 없음 — 시총 가중치 결측")

con.to_csv(RAW_FIRM_DIR / "selected_top10.csv", index=False, encoding="utf-8-sig")
print(f"[구성종목] {len(con)}종목 / 산업별:")
print(con.groupby("industry")["stock_code"].nunique().to_string())


[구성종목] 70종목 / 산업별:
industry
건설업     10
금융업     10
운수장비    10
의약품     10
전기전자    10
철강금속    10
화학      10


### 관찰 결과

- 산업별 구성종목 수와 시총 가중치 결측 여부를 위 출력에서 확인한다.
- 산업당 10종목, 종목코드 6자리·단일 산업 배정이 검증을 통과해야 한다.


## corp_code 매핑

### 목적

- DART 조회에 필요한 corp_code(8자리)를 corpCode.xml에서 매핑한다.
- 삼성전자(005930→00126380)로 매핑 규칙을 점검한다.


In [3]:
# [셀 3] corpCode.xml 다운로드 + stock_code→corp_code 매핑
corp_path = RAW_FIRM_DIR / "corpCode.xml"
if not corp_path.exists():
    try:
        res = requests.get("https://opendart.fss.or.kr/api/corpCode.xml",
                           params={"crtfc_key": API_KEY}, timeout=90)
        res.raise_for_status()
        with zipfile.ZipFile(BytesIO(res.content)) as zf:
            corp_path.write_bytes(zf.read([n for n in zf.namelist() if n.endswith(".xml")][0]))
        print(f"[다운로드] corpCode.xml ({corp_path.stat().st_size/1e6:.1f}MB)")
    except Exception as e:
        raise RuntimeError(f"[중단] corpCode.xml 실패 — 키/네트워크 확인: {e}")
root = ET.parse(corp_path).getroot()
s2c = {}
for c in root.findall(".//list"):
    sc = (c.findtext("stock_code") or "").strip()
    if sc:
        s2c[sc.zfill(6)] = (c.findtext("corp_code") or "").strip()
assert s2c.get("005930") == "00126380", "[중단] corp_code 매핑 규칙 오류"
con["corp_code"] = con["stock_code"].map(s2c).fillna("").str.zfill(8).replace("00000000", "")
n_ok = int((con["corp_code"].str.len() == 8).sum())
print(f"[Sanity] 삼성전자 005930 → 00126380 ✓")
print(f"[corp_code 매핑] {n_ok}/{len(con)} (미매핑은 수집단계 원장 기록)")


[다운로드] corpCode.xml (30.1MB)
[Sanity] 삼성전자 005930 → 00126380 ✓
[corp_code 매핑] 70/70 (미매핑은 수집단계 원장 기록)


### 관찰 결과

- 삼성전자 sanity check 통과와 매핑 성공률을 확인한다.
- 미매핑 종목은 아래 수집 단계의 오류 원장에 기록된다.


## DART 재무·배당·R&D 수집

### 목적

- 기업×연도별로 부채비율·배당수익률·매출·경상연구개발비를 수집한다.
- 기업 단위 try/except로 실패를 격리하고, 원문 저장·배치 체크포인트·재실행 스킵·요청 한도 대응을 적용한다.


In [4]:
# [셀 4] 수집 함수 — 재무제표(부채·자본·매출·경상연구개발비) + 배당수익률
class FatalCollectionError(Exception):
    pass
_FATAL_STATUS = {"010", "011", "012", "020", "021"}
RND_NAMES = ["경상연구개발비", "연구개발비", "경상개발비", "경상연구비"]
SALES_NAMES = ["매출액", "수익(매출액)", "영업수익"]

def _to_num(x):
    try:
        return float(str(x).replace(",", "").strip())
    except (ValueError, AttributeError):
        return np.nan

def _cached_get(url, params, path):
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=25)
        except requests.RequestException:
            time.sleep(0.4 * (attempt + 1)); continue
        # 원문 보존(파싱 실패 시 재요청 불필요)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.with_suffix(".response.txt").write_text(r.text, encoding="utf-8")
        try:
            js = r.json()
        except ValueError:
            time.sleep(0.4 * (attempt + 1)); continue
        tmp = path.with_suffix(".json.tmp")
        tmp.write_text(json.dumps(js, ensure_ascii=False), encoding="utf-8"); tmp.replace(path)
        return js
    return None

def fetch_financials(corp_code, year, api_key):
    acct, used = {}, None
    for fs_div in ("CFS", "OFS"):
        js = _cached_get("https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json",
                         {"crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
                          "reprt_code": "11011", "fs_div": fs_div},
                         DART_RAW_DIR / str(year) / f"{corp_code}_{fs_div}.json")
        if js is None:
            continue
        st = js.get("status")
        if st in _FATAL_STATUS:
            raise FatalCollectionError(f"재무 status={st} ({js.get('message')})")
        if st != "000":
            continue
        for row in js.get("list", []):
            nm = (row.get("account_nm") or "").replace(" ", "")
            amt = _to_num(row.get("thstrm_amount"))
            if nm and nm not in acct and not np.isnan(amt):
                acct[nm] = amt
        if acct:
            used = fs_div; break

    def pick(names):
        for n in names:
            if n in acct:
                return acct[n]
        return np.nan
    equity = acct.get("자본총계", np.nan); liab = acct.get("부채총계", np.nan)
    sales = pick(SALES_NAMES); rnd = pick(RND_NAMES)
    debt_ratio = liab / equity if (not np.isnan(liab) and not np.isnan(equity) and equity != 0) else np.nan
    return {"equity": equity, "liabilities": liab, "sales": sales, "rnd_expense": rnd,
            "debt_ratio": debt_ratio, "fs_div": used, "fin_status": "ok" if acct else "nodata"}

def fetch_dividend_yield(corp_code, year, api_key):
    js = _cached_get("https://opendart.fss.or.kr/api/alotMatter.json",
                     {"crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year), "reprt_code": "11011"},
                     DART_RAW_DIR / str(year) / f"{corp_code}_div.json")
    if js is None:
        return np.nan, "error"
    st = js.get("status")
    if st in _FATAL_STATUS:
        raise FatalCollectionError(f"배당 status={st} ({js.get('message')})")
    if st == "013":
        return (0.0 if DIV_MISSING_AS_ZERO else np.nan), "무배당"
    if st != "000":
        return np.nan, "error"
    cand = []
    for row in js.get("list", []):
        se = (row.get("se") or "").replace(" ", "")
        if "현금배당수익률" in se:
            knd = (row.get("stock_knd") or ""); v = _to_num(row.get("thstrm"))
            cand.append((("보통" in knd) or knd.strip() == "", v))
    numeric = [(p, v) for p, v in cand if not (isinstance(v, float) and np.isnan(v))]
    if not numeric:
        return (0.0 if DIV_MISSING_AS_ZERO else np.nan), "무배당"
    numeric.sort(key=lambda t: (not t[0]))
    return float(numeric[0][1]), "ok"

# 스모크 체크
_r = fetch_financials("00126380", 2023, API_KEY)
_dy, _ds = fetch_dividend_yield("00126380", 2023, API_KEY)
print(f"[스모크] 삼성전자 2023: debt_ratio={_r['debt_ratio']:.3f} · dividend_yield(%)={_dy:.2f} ({_ds})")


[스모크] 삼성전자 2023: debt_ratio=0.254 · dividend_yield(%)=1.90 (ok)


In [ ]:
# [셀 5] 전체 firm-year 수집 — 오류 원장 + 배치 체크포인트 + 재실행 스킵 + 한도 대응
FIN_CSV = RAW_FIRM_DIR / "firm_financials.csv"
ERR_CSV = RAW_FIRM_DIR / "collection_errors.csv"
BATCH, PAUSE = 40, 0.1

def _save(base, rows, errors):
    cur = pd.concat([base, pd.DataFrame(rows)], ignore_index=True) if (len(base) or rows) else pd.DataFrame()
    if len(cur):
        cur = cur.drop_duplicates(["stock_code", "fiscal_year"], keep="last")
        tmp = FIN_CSV.with_suffix(".csv.tmp"); cur.to_csv(tmp, index=False, encoding="utf-8-sig"); tmp.replace(FIN_CSV)
    if errors:
        old = pd.read_csv(ERR_CSV, dtype=str) if ERR_CSV.exists() else pd.DataFrame()
        pd.concat([old, pd.DataFrame(errors)], ignore_index=True).to_csv(ERR_CSV, index=False, encoding="utf-8-sig")
    return cur

base = pd.DataFrame()
done = set()
if FIN_CSV.exists():
    base = pd.read_csv(FIN_CSV, dtype={"stock_code": str}); base["stock_code"] = base["stock_code"].str.zfill(6)
    ok = base[base["fin_status"].isin(["ok", "nodata"])]
    done = set(zip(ok["stock_code"], ok["fiscal_year"].astype(int)))

src = con[con["corp_code"].str.len() == 8]
rows, errors = [], []
for r in con[con["corp_code"].str.len() != 8].itertuples():
    errors.append({"stock_code": r.stock_code, "fiscal_year": "", "step": "corp_code 매핑",
                   "error": "corpCode.xml 미발견", "result": "수집제외"})
total = len(src) * len(YEARS); i = 0
print(f"[재무·배당·R&D 수집] {len(src)}종목 × {len(YEARS)}년 = {total} firm-year (완료 {len(done)} 스킵)")
for c in src.itertuples():
    for y in YEARS:
        i += 1
        if (c.stock_code, y) in done:
            continue
        try:
            fin = fetch_financials(c.corp_code, y, API_KEY)
            dy, ds = fetch_dividend_yield(c.corp_code, y, API_KEY)
        except FatalCollectionError as e:
            errors.append({"stock_code": c.stock_code, "fiscal_year": y, "step": "DART 한도/인증",
                           "error": str(e), "result": "checkpoint 후 중단"})
            _save(base, rows, errors); raise
        except Exception as e:
            fin = {"equity": np.nan, "liabilities": np.nan, "sales": np.nan, "rnd_expense": np.nan,
                   "debt_ratio": np.nan, "fs_div": None, "fin_status": "error"}
            dy, ds = np.nan, "error"
            errors.append({"stock_code": c.stock_code, "fiscal_year": y, "step": "수집예외",
                           "error": str(e)[:200], "result": "오류기록·계속"})
        rows.append({"industry": c.industry, "company_name": c.company_name, "stock_code": c.stock_code,
                     "corp_code": c.corp_code, "fiscal_year": y, **fin,
                     "dividend_yield": dy, "div_status": ds})
        if ds not in ("ok", "무배당"):
            errors.append({"stock_code": c.stock_code, "fiscal_year": y, "step": "배당수집",
                           "error": ds, "result": "배당결측"})
        if len(rows) % BATCH == 0:
            _save(base, rows, errors); print(f"  {i}/{total} checkpoint")
        time.sleep(PAUSE)
FIN = _save(base, rows, errors)
# 개인 PC 절대경로를 출력에 남기지 않는다 (config._rel: 저장소 루트 기준 상대경로).
print(f"[저장] {config._rel(FIN_CSV)} ({len(FIN)}행)")
for col in ["debt_ratio", "dividend_yield", "sales", "rnd_expense"]:
    print(f"  {col:14s} 확보 {int(FIN[col].notna().sum())}/{len(FIN)}")


### 관찰 결과

- 변수별 확보 건수(부채비율·배당수익률·매출·R&D)와 오류 원장 규모를 위 출력에서 기록한다.
- 부채비율은 결측 연도를 제외하고 존재 연도만 평균에 사용한다.
- 배당수익률은 DART 배당 API에서 수치가 확인되지 않은 연도를 원수집 단계에서 0으로 계상한다(`DIV_MISSING_AS_ZERO=True`). 다만 이 상태는 명시적 무배당과 비관측을 완전히 구분하지 못할 수 있으므로, 0 처리 결과를 단독 근거로 사용하지 않는다. STEP10에서는 `div_status=='ok'`인 관측만 남기고 나머지를 결측으로 처리한 대안 사양을 함께 제시해 결론의 민감도를 확인한다.


## 수집 품질 검증

### 목적

- 예상 firm-year 대비 수집 범위, 결측과 실제 0 구분, 부채비율 재계산 일치, inf 여부를 점검한다.


In [6]:
# [셀 6] 수집 결과 검증
assert not FIN.duplicated(["stock_code", "fiscal_year"]).any(), "[설계오류] firm-year 중복"
cov = FIN.groupby("stock_code")["fiscal_year"].nunique()
print(f"[coverage] 기업당 연도 평균 {cov.mean():.1f}/{len(YEARS)} · 최소 {int(cov.min())}")
for c in ["debt_ratio", "dividend_yield", "sales", "rnd_expense"]:
    v = pd.to_numeric(FIN[c], errors="coerce")
    print(f"  {c:14s} 0값 {int((v==0).sum()):4d} · 결측 {int(v.isna().sum()):4d}")
with np.errstate(divide="ignore", invalid="ignore"):
    eq = pd.to_numeric(FIN["equity"], errors="coerce")
    exp = (pd.to_numeric(FIN["liabilities"], errors="coerce") / eq).where(eq != 0)
got = pd.to_numeric(FIN["debt_ratio"], errors="coerce")
m = got.notna() & exp.notna()
assert bool(np.allclose(got[m], exp[m], rtol=1e-6)) if m.any() else True, "[검증실패] 부채비율 재계산 불일치"
assert int(np.isinf(got).sum()) == 0, "[검증실패] 부채비율 inf"
print("[검증] 부채비율 = 부채총계/자본총계 재계산 일치 · inf 없음 ✓")


[coverage] 기업당 연도 평균 10.0/10 · 최소 10
  debt_ratio     0값    0 · 결측  110
  dividend_yield 0값  194 · 결측    0
  sales          0값    0 · 결측  182
  rnd_expense    0값    2 · 결측  621
[검증] 부채비율 = 부채총계/자본총계 재계산 일치 · inf 없음 ✓


### 관찰 결과

- 부채비율 재계산 일치·inf 부재를 확인한다.
- 변수별 실제 0(무배당·저부채)과 결측(미상장·미공시)이 구분되어 기록된다.


## 금융업 부채비율 결측 보완 — DART 사업보고서 원문 재수집

In [ ]:
# [셀 6-1] 금융업 부채비율 결측 보완 — DART 사업보고서 원문 재수집 (XBRL API 공백 구간)
# 배경: fnlttSinglAcntAll·fnlttSinglAcnt 두 API 엔드포인트는 금융지주·은행·보험 8개사의
#       2022년 이전 XBRL 상세 재무제표를 반환하지 않는다(013 조회된 데이타가 없습니다, CFS·OFS 모두 동일).
#       해당 회사들의 사업보고서(정기공시) 자체는 존재하므로, document.xml(원문)에서
#       "요약연결재무정보"의 부채총계·자본총계를 직접 추출해 보완한다(평균대체·보간이 아닌 1차 자료 값 복구).

FIN_CSV = RAW_FIRM_DIR / "firm_financials.csv"
RECOVERY_CSV = RAW_FIRM_DIR / "dart_debt_ratio_recovery.csv"
assert FIN_CSV.exists(), f"[필수입력 없음] {FIN_CSV}"

fin_raw = pd.read_csv(FIN_CSV)
before_na = int(fin_raw.loc[fin_raw["industry"] == "금융업", "debt_ratio"].isna().sum())
n_fin_cells = int((fin_raw["industry"] == "금융업").sum())

if RECOVERY_CSV.exists():
    recovery = pd.read_csv(RECOVERY_CSV)
    fin_recovered = fin_raw.copy()
    filled = 0
    for _, r in recovery.iterrows():
        mask = (fin_recovered["company_name"] == r["company_name"]) & (fin_recovered["fiscal_year"] == r["fiscal_year"])
        if mask.sum() != 1:
            continue
        if fin_recovered.loc[mask, "debt_ratio"].notna().all():
            continue  # 원 수집값이 이미 있으면 절대 덮어쓰지 않는다
        fin_recovered.loc[mask, "liabilities"] = r["liabilities"]
        fin_recovered.loc[mask, "equity"] = r["equity"]
        fin_recovered.loc[mask, "debt_ratio"] = r["debt_ratio"]
        filled += 1
    FIN_RECOVERED_CSV = PROCESSED_DIR / "firm_financials_recovered.csv"
    fin_recovered.to_csv(FIN_RECOVERED_CSV, index=False)
    after_na = int(fin_recovered.loc[fin_recovered["industry"] == "금융업", "debt_ratio"].isna().sum())
    cov = fin_recovered.loc[fin_recovered["industry"] == "금융업"].groupby("company_name")["debt_ratio"].apply(lambda s: s.notna().sum())
    print(f"[부채비율 결측 보완] {RECOVERY_CSV.name}에서 {filled}건 채움")
    print(f"[금융업 debt_ratio 결측] {before_na} -> {after_na}  (금융업 총 {n_fin_cells}칸 中)")
    print(f"[회사별 확보 연도 수 / 10]")
    print(cov.to_string())
    # 개인 PC 절대경로를 출력에 남기지 않는다 (config._rel: 저장소 루트 기준 상대경로).
    print(f"[저장] {config._rel(FIN_RECOVERED_CSV)}")
else:
    fin_recovered = fin_raw
    print(f"[안내] {RECOVERY_CSV.name} 없음 — 보완 생략, 원본 {FIN_CSV.name} 그대로 사용 (결측 {before_na}건)")


### 관찰 결과

- API(`fnlttSinglAcntAll`/`fnlttSinglAcnt`)가 반환하지 못한 8개 금융지주·은행·보험사의 2016~2022년
  부채총계·자본총계는 해당 연도 사업보고서 원문(`document.xml`)의 "요약연결재무정보" 표에서
  직접 추출해 복구했다. 추출값은 이후 회계연도의 API 수집값(2023~2025)과 겹치는 연도에서
  전부 일치를 확인했다(예: KB금융 2021년 부채총계 615,601,914백만원 — 2021 사업보고서·2023
  사업보고서 비교표시 두 출처 모두 동일).
- 우리금융지주는 2019년에 금융지주회사로 신설된 법인이라 2016~2018년 사업보고서 자체가
  존재하지 않는다. 이 3칸은 수집 실패가 아니라 해당 기간에 지금의 법인이 존재하지 않았다는
  구조적 사유이므로 결측으로 남긴다(대체·추정하지 않음).
- 신한지주 2024년은 API 응답에 부채총계는 있으나 자본총계 태그만 누락되어 있어,
  같은 응답의 자본과부채총계(자산총계)-부채총계로 산출했다(1차 응답 내부의 항등식 이용,
  타사·타연도 값을 끌어온 것이 아님).
- 복구는 `data/raw/firm/firm_financials.csv`를 직접 덮어쓰지 않고, 기존 원 수집값이 있는
  칸은 절대 바꾸지 않은 채 `data/processed/firm_financials_recovered.csv`로 별도 저장한다.
  이후 STEP10의 부채비율 관련 계산은 이 보완 파일을 사용하고, 배당수익률·R&D 등 다른 변수는
  영향이 없으므로 원본 `firm_financials.csv`를 그대로 사용한다.


## 의약품 R&D 결합

### 목적

- 의약품 R&D를 매출 대비 비율(R&D intensity)로 정리한다. 비용처리 기업은 DART 경상연구개발비/매출, 자본화 기업은 사업보고서 연구개발비/매출 비율(입력 CSV)을 사용한다.
- 7개 산업 Spearman 변수가 아니라 의약품 표본의 연구개발 지출 특성을 기술하는 보조자료로 사용한다. 현금흐름 듀레이션의 직접 대리변수나 금리베타의 설명변수로 사용하지 않으며, 산업 시총가중·순위상관은 만들지 않는다.


In [7]:
# [셀 7] 의약품 R&D/매출 결합 (API 비용처리분 + 사업보고서 자본화분)
CAP_CSV = KRX_MANUAL_DIR / "rnd_pharma_capitalized.csv"
ph = FIN[FIN["industry"] == "의약품"].copy()
ph["rnd_ratio_api"] = np.where(ph["sales"].notna() & (ph["sales"] != 0),
                               100 * ph["rnd_expense"] / ph["sales"], np.nan)
api_firm = (ph.groupby(["company_name", "stock_code"])
              .agg(rnd_to_sales_pct=("rnd_ratio_api", "mean"),
                   n_years=("rnd_ratio_api", lambda s: int(s.notna().sum()))).reset_index())
api_firm = api_firm[api_firm["n_years"] > 0].copy()
api_firm["기준"] = "DART 경상연구개발비/매출(비용처리분)"

if CAP_CSV.exists():
    cap = pd.read_csv(CAP_CSV, dtype={"stock_code": str}); cap["stock_code"] = cap["stock_code"].str.zfill(6)
    cap_firm = (cap.groupby(["company_name", "stock_code"])["rnd_to_sales_pct"].mean().reset_index())
    cap_firm["기준"] = "사업보고서 연구개발비/매출(연결·총계)"
    cap_firm["n_years"] = cap.groupby(["company_name", "stock_code"])["fiscal_year"].nunique().values
else:
    cap_firm = pd.DataFrame(columns=api_firm.columns)
    print(f"[경고] {CAP_CSV} 없음 — 자본화 기업 R&D 미포함")

# 자본화 입력이 있으면 우선(총계 기준), 없으면 API 값 사용
support = pd.concat([cap_firm, api_firm[~api_firm["stock_code"].isin(cap_firm["stock_code"])]],
                    ignore_index=True).sort_values("rnd_to_sales_pct", ascending=False)
support.to_csv(PROCESSED_DIR / "pharma_rnd_support.csv", index=False, encoding="utf-8-sig")
ph.to_csv(PROCESSED_DIR / "pharma_rnd_firmyear.csv", index=False, encoding="utf-8-sig")
print("[의약품 R&D/매출(%) 보조근거 — 기업별]")
print(support[["company_name", "rnd_to_sales_pct", "기준", "n_years"]].round(2).to_string(index=False))


[의약품 R&D/매출(%) 보조근거 — 기업별]
company_name  rnd_to_sales_pct                     기준  n_years
   SK바이오사이언스             32.03  사업보고서 연구개발비/매출(연결·총계)        3
        셀트리온             15.34  사업보고서 연구개발비/매출(연결·총계)        3
        한미약품             14.67 DART 경상연구개발비/매출(비용처리분)       10
     한올바이오파마             12.53 DART 경상연구개발비/매출(비용처리분)        9
         녹십자             11.63  사업보고서 연구개발비/매출(연결·총계)        3
        대웅제약             11.48 DART 경상연구개발비/매출(비용처리분)        1
         종근당              9.45 DART 경상연구개발비/매출(비용처리분)       10
    삼성바이오로직스              8.77  사업보고서 연구개발비/매출(연결·총계)        3
        유한양행              6.88 DART 경상연구개발비/매출(비용처리분)       10
          대웅              0.01 DART 경상연구개발비/매출(비용처리분)        1


### 관찰 결과

- 의약품 기업별 R&D/매출 비율과 그 출처(API 비용처리분 / 사업보고서 총계)를 위 표에서 기록한다.
- 자본화 4개사(사업보고서 연구개발비 총계 기준)와 비용처리 6개사(DART 경상연구개발비 기준)는 분자 정의가 서로 다르므로, 기업 간 수준 비교는 하지 않는다.


## 해석

- 이 STEP은 H3를 검정하지 않고 STEP10이 사용할 수집 결과 파일만 생성한다.
- 부채비율·배당수익률은 7개 산업 공통, R&D는 의약품 전용 보조근거로 역할을 구분한다.
- 최신 구성종목 고정(2025 스냅샷)과 신규상장 기업의 결측연도는 표본 한계로 남긴다.


## STEP9 요약

### Takeaway

- KOSPI 산업별 지수 상위 10종목에 대해 DART에서 부채비율·배당수익률·매출·R&D를 자동 수집한다.
- 원문 저장·오류 격리·배치 체크포인트·재실행 스킵으로 재현 가능한 수집 구조를 사용한다.
- 배당수익률은 현금배당수익률(보통주), 부채비율은 부채총계/자본총계로 정의한다.
- 의약품 R&D는 매출 대비 비율로만 정리하며 산업 순위상관 변수로 쓰지 않는다.
- 상세 내용: `results/H3_firm_characteristics/summary.md`
